In [40]:
%matplotlib inline
import pandas as pd
import plotly.express as px

# Census data for migration data
# Source: https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html
# data_migration_2024 = "State_to_State_Migration_Table_2024_T13.xlsx"

# FED data for estimated population, census data was more out of date
# Source: https://fred.stlouisfed.org/release/tables?rid=118&eid=259194&od=2024-01-01#
# data_state_2024 = "State_Population_2024.xlsx"

# Census distinction of regions used for graphing and category
# Source: https://www2.census.gov/geo/pdfs/maps-data/maps/reference/us_regdiv.pdf

### Importing Dataset

In [41]:
data_2024_raw = pd.read_excel('Migration_2024_Cleaned.xlsx', sheet_name='Raw_Data', skiprows=1)
# Rows with value 'X' represent not applicable
# Rows with value 'N' means the estimate cannot be displayed because of insufficient number of sample cases
# Replacing both with NaN, also convert all values to numeric
data_2024_raw["Estimated Movers"] = pd.to_numeric(data_2024_raw["Estimated Movers"], errors="coerce")
# data_2024_raw.head()

In [42]:
data_state_2024 = pd.read_excel('State_Population_2024.xlsx', sheet_name='Data_Table')
#data_state_2024.head()

### Summarizing Data

In [43]:
state_abbrev = {'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO','Connecticut':'CT','Delaware':'DE','District of Columbia':'DC','Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY','Puerto Rico':'PR'}

df_2024_summary = pd.DataFrame({
    'State': list(state_abbrev.keys()),
    'Abbr': list(state_abbrev.values()),
    'Inflow': 0,
    'Outflow': 0,
    'Net': 0
    })
#df_2024_summary.head()

In [44]:
# Inflow": how many people are migrating into a state
inflow = data_2024_raw.groupby("Current residence")["Estimated Movers"].sum()
df_2024_summary["Inflow"] = df_2024_summary["State"].map(inflow)
df_2024_summary["Inflow"] = df_2024_summary["Inflow"].fillna(0)

# Outflow: how many people are leaving a state
outflow = data_2024_raw.groupby("Residence 1 year ago")["Estimated Movers"].sum()
df_2024_summary["Outflow"] = df_2024_summary["State"].map(outflow)
df_2024_summary["Outflow"] = df_2024_summary["Outflow"].fillna(0)

# Net Migration
df_2024_summary["Net"] = df_2024_summary["Inflow"] - df_2024_summary["Outflow"]

# print(df_2024_summary.head())
# print(df_2024_summary.sort_values("Net", ascending=False).head())
# print(df_2024_summary.sort_values("Net", ascending=True).head())

In [45]:
# Adding state population to summary
df_2024_summary = df_2024_summary.merge(
    data_state_2024[['State', 'Estimated_Population']],
    on='State',
    how='left'
)

In [46]:
# Creating a per capita migration metric to try and adjust for state population differences
df_2024_summary['Net_per_1000'] = round((
    df_2024_summary['Net'] / df_2024_summary['Estimated_Population']
) * 1000, 2)

# Dropping these two regions as the data is difficult to find and they do not appear using plotly USA map.
df_2024_summary = df_2024_summary[df_2024_summary.State != "District of Columbia"]
df_2024_summary = df_2024_summary[df_2024_summary.State != "Puerto Rico"]

df_2024_summary.head()

,State,Abbr,Inflow,Outflow,Net,Estimated_Population,Net_per_1000
0,Alabama,AL,142143.0,97317.0,44826.0,5163060.0,8.68
1,Alaska,AK,34731.0,39842.0,-5111.0,736537.0,-6.94
2,Arizona,AZ,296143.0,179314.0,116829.0,7556420.0,15.46
3,Arkansas,AR,75034.0,56414.0,18620.0,3096080.0,6.01
4,California,CA,713740.0,662053.0,51687.0,39364770.0,1.31


### Graphing

In [47]:
# For color options https://plotly.com/python/builtin-colorscales/#builtin-sequential-color-scales
fig = px.choropleth(
    df_2024_summary,
    locations="Abbr",
    locationmode="USA-states",
    color="Net",
    scope="usa",
    color_continuous_scale="RdBu",
    # title="Net Migration by State (2024)"
)

# Center color scale at 0
fig.update_layout(
    coloraxis_colorbar_title="Net Migration",
    width=700,
    height=500
)

fig.show()

In [48]:
fig = px.choropleth(
    df_2024_summary,
    locations="Abbr",
    locationmode="USA-states",
    color="Net_per_1000",
    scope="usa",
    color_continuous_scale="RdBu",
    # title="Net Migration by State Per Capita (2024)"
)

# Center color scale at 0
fig.update_layout(
    coloraxis_colorbar_title="Net Migration Per 1000",
    width=700,
    height=500
)

fig.show()

### Summarizing Data Continued...

In [49]:
top_states = df_2024_summary.nlargest(5, 'Net_per_1000')
bottom_states = df_2024_summary.nsmallest(5, 'Net_per_1000')

df_extremes = pd.concat([top_states, bottom_states])

In [50]:
region_map = {
    # Division 1: New England
    'CT': 'New England', 'ME': 'New England', 'MA': 'New England',
    'NH': 'New England', 'RI': 'New England', 'VT': 'New England',

    # Division 2: Middle Atlantic
    'NJ': 'Middle Atlantic', 'NY': 'Middle Atlantic', 'PA': 'Middle Atlantic',

    # Division 3: East North Central
    'IL': 'East North Central', 'IN': 'East North Central', 'MI': 'East North Central',
    'OH': 'East North Central', 'WI': 'East North Central',

    # Division 4: West North Central
    'IA': 'West North Central', 'KS': 'West North Central', 'MN': 'West North Central',
    'MO': 'West North Central', 'NE': 'West North Central',
    'ND': 'West North Central', 'SD': 'West North Central',

    # Division 5: South Atlantic
    'DE': 'South Atlantic', 'FL': 'South Atlantic', 'GA': 'South Atlantic',
    'MD': 'South Atlantic', 'NC': 'South Atlantic', 'SC': 'South Atlantic',
    'VA': 'South Atlantic', 'WV': 'South Atlantic',

    # Division 6: East South Central
    'AL': 'East South Central', 'KY': 'East South Central',
    'MS': 'East South Central', 'TN': 'East South Central',

    # Division 7: West South Central
    'AR': 'West South Central', 'LA': 'West South Central',
    'OK': 'West South Central', 'TX': 'West South Central',

    # Division 8: Mountain
    'AZ': 'Mountain', 'CO': 'Mountain', 'ID': 'Mountain',
    'MT': 'Mountain', 'NV': 'Mountain', 'NM': 'Mountain',
    'UT': 'Mountain', 'WY': 'Mountain',

    # Division 9: Pacific
    'AK': 'Pacific', 'CA': 'Pacific', 'HI': 'Pacific',
    'OR': 'Pacific', 'WA': 'Pacific'
}

df_2024_summary['Region'] = df_2024_summary['Abbr'].map(region_map)

In [51]:
df_region = df_2024_summary.groupby('Region').agg({
    'Net': 'sum',
    'Estimated_Population': 'sum'
}).reset_index()

df_region['Net_per_1000'] = round((
    df_region['Net'] / df_region['Estimated_Population']
) * 1000, 2)

df_region = df_region.sort_values(by='Net_per_1000', ascending=True)

### Graphs Continued...

In [52]:
fig_extremes = px.bar(
    df_extremes,
    x='State',
    y='Net_per_1000',
    color='Net_per_1000',
    color_continuous_scale='RdBu',
    # title='Top and Bottom 5 States by Net Migration per 1,000 (2024)'
)

fig_extremes.update_layout(
    xaxis_title='State',
    yaxis_title='Net Migration per 1,000 Residents',
    coloraxis_showscale=False,
    width=700,
    height=500
)

fig_extremes.show()

In [53]:
fig_region = px.bar(
    df_region,
    x='Region',
    y='Net_per_1000',
    color='Net_per_1000',
    color_continuous_scale='RdBu',
    # title='Net Migration per 1,000 Residents by Census Region (2024)'
)

fig_region.update_layout(
    xaxis_title='Region',
    yaxis_title='Net Migration per 1,000 Residents',
    coloraxis_showscale=False,
    width=700,
    height=500
)

fig_region.show()